# USGS Earthquake Dataset Header Dictionary

This schema represents standard earthquake event data provided by the **USGS (United States Geological Survey)**, annotated with recommendations for **time-series and spatiotemporal model training** (e.g., forecasting, sequence models, seismic hazard prediction).

## Data Summary & ML Training Suitability
| Header | Sample Value | Data Type | Unit / Interpretation | Description | Usable for Time-Series ML? |
|----------|----------|----------|----------|----------|----------|
| `time` | `2025-12-07T02:43:07.342Z` | ISO UTC Timestamp | Exact earthquake origin time in UTC. | Time when rupture began. | **Yes (Primary Time Index)** |
| `latitude` | `29.8698` | Float | 29.8698° North. Range: -90° to +90°. | North-South geographic position. | **Yes (Core Spatial Feature)** |
| `longitude` | `80.9384` | Float | 80.9384° East. Range: -180° to +180°. | East-West geographic position. | **Yes (Core Spatial Feature)** |
| `depth` | `43.404` | Float (km) | Earthquake started 43.404 km below the surface. | Hypocenter depth. | **Yes (Core Spatial Feature)** |
| `mag` | `4.4` | Float | Magnitude 4.4 earthquake. +1 magnitude ≈ 31.6× more energy. | Earthquake size/strength. | **Yes (Target Variable / Key Feature)** |
| `magType` | `mb` | String | Body-Wave Magnitude scale. Other values: `Mw`, `ML`, `Ms`. | Method used to compute magnitude. | **Optional (Categorical)** |
| `nst` | `21` | Integer | 21 seismic stations contributed to location estimation. | Station count used for location. | **Optional (Quality Weighting)** |
| `gap` | `166` | Float (degrees) | Largest angular gap between stations around the epicenter. Smaller is better. | Network coverage quality metric. | **Optional (Quality Weighting)** |
| `dmin` | `8.856` | Float (degrees) | Nearest station was ~983 km away (1° ≈ 111 km). | Distance to closest station. | **Optional (Quality Weighting)** |
| `rms` | `0.94` | Float (seconds) | Average travel-time prediction error of 0.94 s. Lower is better. | Model fit quality. | **Optional (Quality Weighting)** |
| `horizontalError` | `13.18` | Float (km) | Epicenter location uncertainty ≈ 13.18 km. | Horizontal location confidence. | **Optional (Uncertainty Weighting)** |
| `depthError` | `9.35` | Float (km) | Depth estimate uncertainty ≈ 9.35 km. | Depth confidence metric. | **Optional (Uncertainty Weighting)** |
| `magError` | `0.144` | Float | Magnitude uncertainty ±0.144. | Magnitude confidence metric. | **Optional (Uncertainty Weighting)** |
| `magNst` | `14` | Integer | 14 stations used specifically for magnitude calculation. | Magnitude measurement quality. | **Optional (Quality Weighting)** |
| `type` | `earthquake` | String | Event classification. | Earthquake, explosion, quarry blast, etc. | **Filter Only** |
| `status` | `reviewed` | String | Reviewed by a human seismologist. | Verification status. | **Filter Only** |
| `net` | `us` | String | USGS reporting network identifier. | Administrative metadata. | **No (Drop)** |
| `id` | `us7000rkd1` | String | Unique event identifier only. | Arbitrary database ID. | **No (Drop)** |
| `updated` | `2025-12-26T01:20:25.040Z` | ISO UTC Timestamp | Record modification timestamp. | Last update time. | **No (Drop — Leakage Risk)** |
| `place` | `37 km E of Dārchulā...` | String | Human-readable location description. | Textual location. | **No (Drop)** |
| `locationSource` | `us` | String | Agency that computed location. | Administrative metadata. | **No (Drop)** |
| `magSource` | `us` | String | Agency that computed magnitude. | Administrative metadata. | **No (Drop)** |

In [49]:
# CONSTANTS
ORGINAL_DATA_PATH = 'data/nepal_eq_90_26_raw.csv'
FILTERED_DATA_PATH = 'data/nepal_eq_90_26_filtered.csv'
INITIAL_EXTRACTION_DATA_PATH = 'data/nepal_eq_90_26_initial_extraction.csv'

def INITIAL_EXTRACTION_DATA_PATH_TEMPLATE(version: str) -> str:
	return f'data/nepal_eq_90_26_initial_extraction_v{version}.csv'

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_original = pd.read_csv(ORGINAL_DATA_PATH)


# Filtering the dataset to include only relevant features
- Removing all the features that are not relevant to the analysis. The relevant features are mentioned 
in the Features class below. The filtered dataset is saved as 'data/nepal_eq_90_26_filtered.csv'.
and the rest columns are dropped since they are metadata about earthquate and the source of the data.

In [51]:
from enum import Enum
class Features(Enum):
	TIME = 'time'
	LATITUDE = 'latitude'
	LONGITUDE = 'longitude'
	DEPTH = 'depth'
	MAGNITUDE = 'mag'
	MAGNITUDE_TYPE = 'magType'
	NSTATIONS = 'nst'
	GAP = 'gap'
	DMIN = 'dmin'
	RMS = 'rms'
	HORIZONTAL_ERROR = 'horizontalError'
	DEPTH_ERROR = 'depthError'
	MAGNITUDE_ERROR = 'magError'
	MAGNITUDE_NSTATIONS = 'magNst'	

	DISTRICT = 'district'

df_filtered = df_original[[f.value for f in Features if f != Features.DISTRICT]]
df_filtered.head()

df_filtered.to_csv(FILTERED_DATA_PATH, index=False)

## Radius Finding V1

In [52]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import lambertw

def compute_felt_radius_v1(
	magnitude: float,
	depth_km: float,
	target_mmi: float = 2.5,
	region: str = "active"
) -> float:
	"""
	Computes surface felt radius R_felt (km): the epicentral distance at which
	ground shaking is predicted to reach `target_mmi`, using a regional
	tectonic attenuation model.
	"""
	# 1. Master Configuration Dictionaries
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}

	# 2. Extract settings with safety defaults
	c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])

	# 3. Finite-fault saturation term
	R0 = 10 ** (-0.281 + 0.251 * magnitude)

	# 4. Lambert W setup for 3D effective distance
	a = c3 / c4
	b = (c1 + c2 * magnitude - target_mmi) / c4

	ln10 = np.log(10)
	z = (ln10 / a) * (10.0 ** (b / a))
	R_eff = (a / ln10) * lambertw(z).real

	# 5. Pythagorean depth / finite-fault correction to surface distance
	rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

	if rad_sq <= 0:
		return 0.0

	return float(np.sqrt(rad_sq))


# --- Executing Test ---
m, h, target = 7.8, 8.22, 3
radius = compute_felt_radius_v1(m, h, target_mmi=target, region="active")
print(f"--- MMI\u2265{target} Felt Radius for Mag {m} Event (Depth: {h}km) ---")
print(f"-> Region: active -> Radius: {radius:.1f} km\n")

--- MMI≥3 Felt Radius for Mag 7.8 Event (Depth: 8.22km) ---
-> Region: active -> Radius: 672.1 km



In [53]:
@dataclass(frozen=True)
class FeltRadiusConfigV1:
	"""
	Configures the parameters for felt radius calculation.
	Allows customization of input/output column names and region type.
	`i_` prefix denotes input columns, and `o_` prefix denotes output columns.
	"""
	i_region: str = "active"
	i_time_column: str = "time"
	i_mag_column: str = "mag"
	i_depth_column: str = "depth"
	o_felt_radius_column: str = "felt_radius_km"
	o_felt_area_column: str = "felt_area_sqkm"
	o_max_nearfield_mmi_column: str = "max_nearfield_mmi"


class FeltRadiusCalculatorV1:
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}
	supported_tectonic_regions = set(coefficient_configurations.keys())

	@staticmethod
	def extract_seismic_features(
		df: pd.DataFrame, 
		felt_mmi: float = 2.5, 
		config: FeltRadiusConfigV1 = FeltRadiusConfigV1()
	) -> pd.DataFrame:
		"""
		Extracts physical felt-vibration features from earthquake catalog data
		incorporating regional tectonic attenuation.
		"""
		data = df.copy()
		
		# 1. Fetch and validate regional configurations
		region = config.i_region
		
		if region in FeltRadiusCalculatorV1.coefficient_configurations:
			c1, c2, c3, c4 = FeltRadiusCalculatorV1.coefficient_configurations[region]
		else:
			raise ValueError(f"Unknown region profile: {region}, supported: {FeltRadiusCalculatorV1.supported_tectonic_regions}")
			
		M = data[config.i_mag_column].values
		h = data[config.i_depth_column].fillna(10.0).values  # Default to 10km depth if missing
		
		# 2. Near-field finite fault saturation term R0(M)
		R0 = 10 ** (-0.281 + 0.251 * M)
		
		# 3. Transformed Lambert W parameters
		a = c3 / c4
		b = (c1 + c2 * M - felt_mmi) / c4
		
		# 4. Vectorized Lambert W evaluation for R_eff across DataFrame arrays
		ln10 = np.log(10)
		z = (ln10 / a) * (10.0 ** (b / a))
		R_eff = (a / ln10) * lambertw(z).real
		
		# 5. Surface felt radius accounting for depth and saturation
		rad_sq = R_eff**2 - R0**2 - h**2
		r_felt = np.sqrt(np.maximum(rad_sq, 0.0))
		
		# 6. Append physical feature columns
		data[config.o_felt_radius_column] = r_felt
		data[config.o_felt_area_column] = np.pi * (r_felt ** 2)
		
		# 7. Peak intensity directly above hypocenter (R_jb = 0)
		R_eff_epicenter = np.sqrt(R0**2 + h**2)
		base_max_mmi = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
		data[config.o_max_nearfield_mmi_column] = base_max_mmi
		
		return data


config = FeltRadiusConfigV1(
	i_region="active",
	i_time_column="time",
	i_mag_column="mag",
	i_depth_column="depth",
	o_felt_radius_column="felt_radius_km",
	o_felt_area_column="felt_area_sqkm",
	o_max_nearfield_mmi_column="max_nearfield_mmi"
)
mmi = 4.0
df_extracted_1 = FeltRadiusCalculatorV1.extract_seismic_features(
	df_filtered, 
	felt_mmi=mmi, 
	config=config
)
df_extracted_1.to_csv(INITIAL_EXTRACTION_DATA_PATH_TEMPLATE("1"), index=False)

In [54]:
earthquake_catalog = [
    {
        "name": "2015 Gorkha (Nepal)",
        "magnitude": 7.8,
        "depth_km": 15.0,
        "region": "active",
        "observed_felt_radius_km": "~800 - 1000",
        "notes": "Felt widely across Northern India, Tibet, and Bangladesh"
    },
    {
        "name": "2011 Mineral (Virginia, USA)",
        "magnitude": 5.8,
        "depth_km": 6.0,
        "region": "stable_cena",
        "observed_felt_radius_km": "~800 - 1000",
        "notes": "Very low attenuation in CENA; felt in NYC, Toronto & Atlanta"
    },
    {
        "name": "2011 Tohoku (Japan)",
        "magnitude": 9.1,
        "depth_km": 29.0,
        "region": "subduction",
        "observed_felt_radius_km": "> 1200",
        "notes": "Megathrust event; shaking felt throughout Honshu and eastern China"
    },
    {
        "name": "2023 Kahramanmaraş (Turkey)",
        "magnitude": 7.8,
        "depth_km": 10.0,
        "region": "strike_slip",
        "observed_felt_radius_km": "~700 - 900",
        "notes": "Major strike-slip rupture; felt in Syria, Cyprus, Egypt & Iraq"
    },
    {
        "name": "1989 Loma Prieta (California)",
        "magnitude": 6.9,
        "depth_km": 19.0,
        "region": "strike_slip",
        "observed_felt_radius_km": "~350 - 450",
        "notes": "San Andreas strike-slip; higher active crust attenuation"
    },
    {
        "name": "2021 Fagradalsfjall (Iceland)",
        "magnitude": 5.7,
        "depth_km": 5.0,
        "region": "volcanic",
        "observed_felt_radius_km": "~100 - 150",
        "notes": "High thermal attenuation in rift zone limits long-range shaking"
    },
    {
        "name": "2010 Christchurch (New Zealand)",
        "magnitude": 6.2,
        "depth_km": 5.0,
        "region": "active",
        "observed_felt_radius_km": "~250 - 350",
        "notes": "Shallow active crustal event felt across South Island"
    }
]

# Run validation across catalog
results = []
TARGET_MMI = 2.5  # Typical human perception threshold (MMI II-III)

for eq in earthquake_catalog:
    predicted_radius = compute_felt_radius_v1(
        magnitude=eq["magnitude"],
        depth_km=eq["depth_km"],
        target_mmi=TARGET_MMI,
        region=eq["region"]
    )
    predicted_area = np.pi * (predicted_radius ** 2)
    
    results.append({
        "Earthquake Event": eq["name"],
        "Mag (Mw)": eq["magnitude"],
        "Depth (km)": eq["depth_km"],
        "Region": eq["region"],
        "Predicted Radius (km)": round(predicted_radius, 1),
        "Predicted Area (sq km)": f"{predicted_area:,.0f}",
        "Observed USGS Felt Radius (km)": eq["observed_felt_radius_km"]
    })

df_validation = pd.DataFrame(results)

print(f"=== EARTHQUAKE FELT RADIUS VALIDATION (Target MMI = {TARGET_MMI}) ===")
print(df_validation.to_string(index=False))

=== EARTHQUAKE FELT RADIUS VALIDATION (Target MMI = 2.5) ===
               Earthquake Event  Mag (Mw)  Depth (km)      Region  Predicted Radius (km) Predicted Area (sq km) Observed USGS Felt Radius (km)
            2015 Gorkha (Nepal)       7.8        15.0      active                  833.2              2,181,150                    ~800 - 1000
   2011 Mineral (Virginia, USA)       5.8         6.0 stable_cena                 1433.9              6,459,068                    ~800 - 1000
            2011 Tohoku (Japan)       9.1        29.0  subduction                 1629.9              8,345,919                         > 1200
    2023 Kahramanmaraş (Turkey)       7.8        10.0 strike_slip                  604.9              1,149,629                     ~700 - 900
  1989 Loma Prieta (California)       6.9        19.0 strike_slip                  392.5                484,002                     ~350 - 450
  2021 Fagradalsfjall (Iceland)       5.7         5.0    volcanic                

## Radius Finding V2

In [61]:
import numpy as np
from scipy.special import lambertw

def compute_felt_radius_v2(
	magnitude: float,
	depth_km: float,
	target_mmi: float = 2.5,
	region: str = "active",
	soil_type: str = "solid_rock"
) -> float:
	"""
	Computes surface felt radius R_felt (km): the epicentral distance at which
	ground shaking is predicted to reach `target_mmi`, combining a regional
	tectonic attenuation model with a local soil/rock (Vs30) amplification term.
	"""
	# 1. Master Configuration Dictionaries
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}

	soil_configurations = {
		"hard_rock":   (1500.0, "Hard bedrock"),
		"solid_rock":  (760.0,  "Standard baseline engineering rock"),
		"soft_rock":   (560.0,  "Dense soil / weathered rock"),
		"stiff_soil":  (270.0,  "Stiff urban soil"),
		"soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
		"swamp_mud":   (100.0,  "Ultra-soft marsh muds")
	}

	# 2. Extract settings with safety defaults
	c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
	vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

	# 3. Site amplification relative to 760 m/s baseline rock
	v_ref = 760.0
	b_site = 1.8
	delta_mmi_site = b_site * np.log10(v_ref / vs30)

	# Back out the site term from the target: raises the required base-rock
	# intensity on hard rock (delta<0), lowers it on soft soil (delta>0)
	adjusted_target_mmi = target_mmi - delta_mmi_site

	# 4. Finite-fault saturation term
	R0 = 10 ** (-0.281 + 0.251 * magnitude)

	# 5. Lambert W setup for 3D effective distance
	a = c3 / c4
	b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

	ln10 = np.log(10)
	z = (ln10 / a) * (10.0 ** (b / a))
	R_eff = (a / ln10) * lambertw(z).real

	# 6. Pythagorean depth / finite-fault correction to surface distance
	rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

	if rad_sq <= 0:
		return 0.0

	return float(np.sqrt(rad_sq))

# --- Executing a Multivariable Grid Test ---
m, h, target = 7.8, 8.22, 3

print(f"--- MMI\u2265{target} Felt Radius Boundaries for a Mag {m} Event (Depth: {h}km) ---")
for soil in ["hard_rock", "solid_rock", "stiff_soil", "soft_basin"]:
	radius = compute_felt_radius_v2(m, h, target_mmi=target, region="active", soil_type=soil)
	print(f"-> Region: active | Rock Profile: {soil:<12} -> Radius: {radius:.1f} km")

--- MMI≥3 Felt Radius Boundaries for a Mag 7.8 Event (Depth: 8.22km) ---
-> Region: active | Rock Profile: hard_rock    -> Radius: 516.7 km
-> Region: active | Rock Profile: solid_rock   -> Radius: 672.1 km
-> Region: active | Rock Profile: stiff_soil   -> Radius: 939.2 km
-> Region: active | Rock Profile: soft_basin   -> Radius: 1104.1 km


In [56]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import lambertw


@dataclass(frozen=True)
class FeltRadiusConfigV2:
	"""
	Configures the parameters for felt radius calculation.
	Allows customization of input/output column names, region type, and soil profile.
	Here, `i_` prefix denotes input columns, and `o_` prefix denotes output columns.
	Args:
		i_region: str - Tectonic region type (e.g., "active", "stable", "subduction", etc.)
		i_soil_type: str - Local soil/rock profile (e.g., "solid_rock", "soft_basin", etc.)
		i_time_column: str - Name of the time column in the DataFrame
		i_mag_column: str - Name of the magnitude column in the DataFrame
		i_depth_column: str - Name of the depth column in the DataFrame
		o_felt_radius_column: str - Name of the output felt radius column to be added to the DataFrame
		o_felt_area_column: str - Name of the output felt area column to be added to the DataFrame
		o_max_nearfield_mmi_column: str - Name of the output max near-field MMI column to be added to the DataFrame
	"""
	i_region: str = "active"
	i_soil_type: str = "solid_rock" 
	i_time_column: str = "time"
	i_mag_column: str = "mag"
	i_depth_column: str = "depth"
	o_felt_radius_column: str = "felt_radius_km"
	o_felt_area_column: str = "felt_area_sqkm"
	o_max_nearfield_mmi_column: str = "max_nearfield_mmi"


class FeltRadiusCalculatorV2:
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}
	supported_tectonic_regions = set(coefficient_configurations.keys())
	soil_configurations = {
		"hard_rock":   (1500.0, "Hard bedrock"),
		"solid_rock":  (760.0,  "Standard baseline engineering rock"),
		"soft_rock":   (560.0,  "Dense soil / weathered rock"),
		"stiff_soil":  (270.0,  "Stiff urban soil"),
		"soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
		"swamp_mud":   (100.0,  "Ultra-soft marsh muds")
	}
	supported_soil_profiles = set(soil_configurations.keys())

	@staticmethod
	def extract_seismic_features(
		df: pd.DataFrame, 
		felt_mmi: float = 2.5, 
		config: FeltRadiusConfigV2 = FeltRadiusConfigV2()
	) -> pd.DataFrame:
		"""
		Extracts physical felt-vibration features from earthquake catalog data
		incorporating tectonic attenuation and Vs30 local soil amplification.
		"""
		data = df.copy()
		
		# 1. Fetch and validate regional & soil configurations
		region = config.i_region
		soil_type = config.i_soil_type
		
		if region in FeltRadiusCalculatorV2.coefficient_configurations:
			c1, c2, c3, c4 = FeltRadiusCalculatorV2.coefficient_configurations[region]
		else:
			raise ValueError(f"Unknown region profile: {region}, supported: {FeltRadiusCalculatorV2.supported_tectonic_regions}")
			
		if soil_type in FeltRadiusCalculatorV2.soil_configurations:
			vs30, _ = FeltRadiusCalculatorV2.soil_configurations[soil_type]
		else:
			raise ValueError(f"Unknown soil profile: {soil_type}, supported: {FeltRadiusCalculatorV2.supported_soil_profiles}")
			
		M = data[config.i_mag_column].values
		h = data[config.i_depth_column].fillna(10.0).values  # Default to 10km depth if missing
		
		# 2. Site Amplification Correction (Vs30 vs 760m/s baseline)
		v_ref = 760.0
		b_site = 1.8
		delta_mmi_site = b_site * np.log10(v_ref / vs30)
		
		# Adjust target baseline intensity downward for soil amplification
		adjusted_target_mmi = felt_mmi - delta_mmi_site
		
		# 3. Near-field finite fault saturation term R0(M)
		R0 = 10 ** (-0.281 + 0.251 * M)
		
		# 4. Transformed Lambert W parameters
		a = c3 / c4
		b = (c1 + c2 * M - adjusted_target_mmi) / c4
		
		# 5. Vectorized Lambert W evaluation for R_eff across DataFrame arrays
		ln10 = np.log(10)
		z = (ln10 / a) * (10.0 ** (b / a))
		R_eff = (a / ln10) * lambertw(z).real
		
		# 6. Surface felt radius accounting for depth and saturation
		rad_sq = R_eff**2 - R0**2 - h**2
		r_felt = np.sqrt(np.maximum(rad_sq, 0.0))
		
		# 7. Append physical feature columns
		data[config.o_felt_radius_column] = r_felt
		data[config.o_felt_area_column] = np.pi * (r_felt ** 2)
		
		# 8. Peak intensity directly above hypocenter (R_jb = 0) with site correction
		R_eff_epicenter = np.sqrt(R0**2 + h**2)
		base_max_mmi = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
		data[config.o_max_nearfield_mmi_column] = base_max_mmi + delta_mmi_site
		
		return data



config = FeltRadiusConfigV2(
	i_region="active",
	i_soil_type="soft_basin",
	i_time_column="time",
	i_mag_column="mag",
	i_depth_column="depth",
	o_felt_radius_column="felt_radius_km",
	o_felt_area_column="felt_area_sqkm",
	o_max_nearfield_mmi_column="max_nearfield_mmi"
)

mmi = 4.0
data_with_felt_features = FeltRadiusCalculatorV2.extract_seismic_features(
	df_filtered, 
	felt_mmi=mmi, 
	config=config
)

data_with_felt_features.to_csv(INITIAL_EXTRACTION_DATA_PATH_TEMPLATE("2"), index=False)

In [60]:
earthquake_catalog = [
    {
        "name": "2015 Gorkha (Nepal)",
        "magnitude": 7.8,
        "depth_km": 15.0,
        "region": "active",
        "local_soil": "soft_basin",  # Gangetic Plain / Kathmandu basin alluvium
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Mineral (Virginia, USA)",
        "magnitude": 5.8,
        "depth_km": 6.0,
        "region": "stable_cena",
        "local_soil": "stiff_soil",  # Atlantic coastal plain sediments
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Tohoku (Japan)",
        "magnitude": 9.1,
        "depth_km": 29.0,
        "region": "subduction",
        "local_soil": "soft_basin",  # Kanto Plain / coastal basin sediments
        "observed_felt_radius_km": "> 1200"
    },
    {
        "name": "2023 Kahramanmaraş (Turkey)",
        "magnitude": 7.8,
        "depth_km": 10.0,
        "region": "strike_slip",
        "local_soil": "stiff_soil",  # Sedimentary basins across S. Turkey/Syria
        "observed_felt_radius_km": "~700 - 900"
    },
    {
        "name": "1989 Loma Prieta (California)",
        "magnitude": 6.9,
        "depth_km": 19.0,
        "region": "strike_slip",
        "local_soil": "soft_basin",  # SF Bay muds / Marina district alluvium
        "observed_felt_radius_km": "~350 - 450"
    },
    {
        "name": "2021 Fagradalsfjall (Iceland)",
        "magnitude": 5.7,
        "depth_km": 5.0,
        "region": "volcanic",
        "local_soil": "hard_rock",  # Dense basaltic lava bedrock
        "observed_felt_radius_km": "~100 - 150"
    },
    {
        "name": "2010 Christchurch (New Zealand)",
        "magnitude": 6.2,
        "depth_km": 5.0,
        "region": "active",
        "local_soil": "soft_basin",  # Canterbury alluvial gravels/silt
        "observed_felt_radius_km": "~250 - 350"
    }
]

TARGET_MMI = 2.5
results = []

for eq in earthquake_catalog:
    # 1. Baseline Engineering Rock (Vs30 = 760 m/s)
    r_rock = compute_felt_radius_v2(
        eq["magnitude"], eq["depth_km"], TARGET_MMI, eq["region"], soil_type="solid_rock"
    )
    
    # 2. Local Soil Profile Amplified Radius
    r_soil = compute_felt_radius_v2(
        eq["magnitude"], eq["depth_km"], TARGET_MMI, eq["region"], soil_type=eq["local_soil"]
    )
    
    results.append({
        "Earthquake Event": eq["name"],
        "Mag": eq["magnitude"],
        "Region": eq["region"],
        "Site Profile": eq["local_soil"],
        "Rock Radius (km)": round(r_rock, 1),
        "Soil Radius (km)": round(r_soil, 1),
        "Observed USGS Radius (km)": eq["observed_felt_radius_km"]
    })

df_results = pd.DataFrame(results)

print(f"=== EARTHQUAKE FELT RADIUS VALIDATION WITH SOIL AMPLIFICATION (Target MMI = {TARGET_MMI}) ===")
print(df_results.to_string(index=False))

=== EARTHQUAKE FELT RADIUS VALIDATION WITH SOIL AMPLIFICATION (Target MMI = 2.5) ===
               Earthquake Event  Mag      Region Site Profile  Rock Radius (km)  Soil Radius (km) Observed USGS Radius (km)
            2015 Gorkha (Nepal)  7.8      active   soft_basin             833.2             929.9               ~800 - 1000
   2011 Mineral (Virginia, USA)  5.8 stable_cena   stiff_soil            1433.9            1680.8               ~800 - 1000
            2011 Tohoku (Japan)  9.1  subduction   soft_basin            1629.9            1792.2                    > 1200
    2023 Kahramanmaraş (Turkey)  7.8 strike_slip   stiff_soil             604.9             655.9                ~700 - 900
  1989 Loma Prieta (California)  6.9 strike_slip   soft_basin             392.5             452.1                ~350 - 450
  2021 Fagradalsfjall (Iceland)  5.7    volcanic    hard_rock             122.0             110.5                ~100 - 150
2010 Christchurch (New Zealand)  6.2      activ

In [58]:
# import numpy as np
# import plotly.graph_objects as go

# def generate_circle_perimeter(lat: float, lon: float, radius_km: float, num_points: int = 40):
#     """Calculates geodesic perimeter lat/lon coordinates on a sphere."""
#     if radius_km <= 0:
#         return [], []
#     R = 6371.0  # Earth radius in km
#     angles = np.linspace(0, 2 * np.pi, num_points)
#     lat_rad, lon_rad = np.radians(lat), np.radians(lon)
#     d_R = radius_km / R

#     circle_lats = np.arcsin(
#         np.sin(lat_rad) * np.cos(d_R) + 
#         np.cos(lat_rad) * np.sin(d_R) * np.cos(angles)
#     )
#     circle_lons = lon_rad + np.arctan2(
#         np.sin(angles) * np.sin(d_R) * np.cos(lat_rad),
#         np.cos(d_R) - np.sin(lat_rad) * np.sin(circle_lats)
#     )
#     return np.degrees(circle_lats), np.degrees(circle_lons)


# fig = go.Figure()

# # 1. Epicenter points using go.Scattermap
# fig.add_trace(go.Scattermap(
#     lat=data_with_felt_features["latitude"],
#     lon=data_with_felt_features["longitude"],
#     mode="markers",
#     marker=dict(size=7, color="black"),
#     name="Epicenters",
#     text=data_with_felt_features["mag"],
#     hovertemplate="<b>Mag: %{text}</b><br>Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<extra></extra>"
# ))

# # 2. True-scale physical circles
# for _, row in data_with_felt_features.iterrows():
#     r_km = row["felt_radius_km"]
#     if r_km > 0:
#         c_lats, c_lons = generate_circle_perimeter(row["latitude"], row["longitude"], r_km)
		
#         fig.add_trace(go.Scattermap(
#             lat=c_lats,
#             lon=c_lons,
#             mode="lines",
#             fill="toself",
#             fillcolor="rgba(239, 68, 68, 0.20)",
#             line=dict(color="darkred", width=1.5),
#             showlegend=False,
#             hoverinfo="skip"
#         ))

# # 3. Layout uses map= instead of mapbox=
# fig.update_layout(
#     map=dict(
#         style="open-street-map",  # Options: 'open-street-map', 'carto-positron', 'carto-darkmatter'
#         center=dict(lat=28.3949, lon=84.1240),
#         zoom=6.2,
#     ),
#     margin={"r": 0, "t": 40, "l": 0, "b": 0},
#     title="Nepal Earthquakes — Geodesic Felt Extent (km) Overlay"
# )

# fig.show()

In [62]:
def calculate_site_amplification(
    vs30: float, 
    v_ref: float = 760.0, 
    b_site: float = 0.5, 
    upper_limit: float = 0.4, 
    lower_limit: float = -0.8, 
    method: str = "tanh"
) -> float:
    """
    Calculates MMI site amplification using continuous scaling functions.
    Supports both scalar values and NumPy arrays.
    """
    delta_raw = b_site * np.log10(v_ref / vs30)

    if method == "clip":
        # Legacy hard clipping (contains kink points / non-differentiable boundaries)
        delta_site = np.clip(delta_raw, lower_limit, upper_limit)

    elif method == "tanh":
        # Asymmetric smooth hyperbolic tangent saturation (C-infinity continuous)
        abs_lower = abs(lower_limit)
        delta_site = np.where(
            delta_raw >= 0,
            upper_limit * np.tanh(delta_raw / upper_limit),
            -abs_lower * np.tanh(np.abs(delta_raw) / abs_lower)
        )

    elif method == "sigmoid":
        # Generalized continuous logistic scaling
        span = upper_limit - lower_limit
        delta_site = lower_limit + span / (1.0 + np.exp(-4.0 * delta_raw / span))

    else:
        raise ValueError(f"Unknown site scaling method: {method}")

    return float(delta_site) if np.ndim(delta_site) == 0 else delta_site


def compute_felt_radius_v3(
    magnitude: float,
    depth_km: float,
    target_mmi: float = 2.5,
    region: str = "active",
    soil_type: str = "solid_rock",
    site_scaling_method: str = "tanh"
) -> float:
    """
    Computes surface felt radius R_felt (km): the epicentral distance at which
    ground shaking is predicted to reach `target_mmi`, combining a regional
    tectonic attenuation model with a local soil/rock (Vs30) continuous site amplification term.
    """
    # 1. Master Configuration Dictionaries
    coefficient_configurations = {
        "active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
        "stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
        "stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
        "subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
        "strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
        "volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
    }

    soil_configurations = {
        "hard_rock":   (1500.0, "Hard bedrock"),
        "solid_rock":  (760.0,  "Standard baseline engineering rock"),
        "soft_rock":   (560.0,  "Dense soil / weathered rock"),
        "stiff_soil":  (270.0,  "Stiff urban soil"),
        "soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
        "swamp_mud":   (100.0,  "Ultra-soft marsh muds")
    }

    # 2. Extract settings with safety defaults
    c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
    vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

    # 3. Continuous site amplification relative to 760 m/s baseline rock
    delta_mmi_site = calculate_site_amplification(
        vs30=vs30,
        v_ref=760.0,
        b_site=0.5,
        upper_limit=0.4,
        lower_limit=-0.8,
        method=site_scaling_method
    )

    adjusted_target_mmi = target_mmi - delta_mmi_site

    # 4. Finite-fault saturation term
    R0 = 10 ** (-0.281 + 0.251 * magnitude)

    # 5. Lambert W setup for 3D effective distance
    a = c3 / c4
    b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

    ln10 = np.log(10)
    z = (ln10 / a) * (10.0 ** (b / a))
    R_eff = (a / ln10) * lambertw(z).real

    # 6. Pythagorean depth / finite-fault correction to surface distance
    rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

    if rad_sq <= 0:
        return 0.0

    return float(np.sqrt(rad_sq))


# --- Executing Test Across All Rock/Soil Profiles ---
m, h, target = 7.8, 8.22, 3

print(f"--- MMI\u2265{target} Felt Radius Boundaries for a Mag {m} Event (Depth: {h}km) ---")
test_soils = ["hard_rock", "solid_rock", "soft_rock", "stiff_soil", "soft_basin", "swamp_mud"]

for soil in test_soils:
    radius = compute_felt_radius_v3(
        magnitude=m, 
        depth_km=h, 
        target_mmi=target, 
        region="active", 
        soil_type=soil, 
        site_scaling_method="tanh"
    )
    print(f"-> Region: active | Rock Profile: {soil:<12} -> Radius: {radius:.1f} km")

--- MMI≥3 Felt Radius Boundaries for a Mag 7.8 Event (Depth: 8.22km) ---
-> Region: active | Rock Profile: hard_rock    -> Radius: 627.7 km
-> Region: active | Rock Profile: solid_rock   -> Radius: 672.1 km
-> Region: active | Rock Profile: soft_rock    -> Radius: 692.5 km
-> Region: active | Rock Profile: stiff_soil   -> Radius: 736.2 km
-> Region: active | Rock Profile: soft_basin   -> Radius: 761.7 km
-> Region: active | Rock Profile: swamp_mud    -> Radius: 773.9 km


In [65]:
earthquake_catalog = [
    {
        "name": "2015 Gorkha (Nepal)",
        "magnitude": 7.8,
        "depth_km": 15.0,
        "region": "active",
        "local_soil": "soft_basin",  # Gangetic Plain / Kathmandu basin alluvium
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Mineral (Virginia, USA)",
        "magnitude": 5.8,
        "depth_km": 6.0,
        "region": "stable_cena",
        "local_soil": "stiff_soil",  # Atlantic coastal plain sediments
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Tohoku (Japan)",
        "magnitude": 9.1,
        "depth_km": 29.0,
        "region": "subduction",
        "local_soil": "soft_basin",  # Kanto Plain / coastal basin sediments
        "observed_felt_radius_km": "> 1200"
    },
    {
        "name": "2023 Kahramanmaraş (Turkey)",
        "magnitude": 7.8,
        "depth_km": 10.0,
        "region": "strike_slip",
        "local_soil": "stiff_soil",  # Sedimentary basins across S. Turkey/Syria
        "observed_felt_radius_km": "~700 - 900"
    },
    {
        "name": "1989 Loma Prieta (California)",
        "magnitude": 6.9,
        "depth_km": 19.0,
        "region": "strike_slip",
        "local_soil": "soft_basin",  # SF Bay muds / Marina district alluvium
        "observed_felt_radius_km": "~350 - 450"
    },
    {
        "name": "2021 Fagradalsfjall (Iceland)",
        "magnitude": 5.7,
        "depth_km": 5.0,
        "region": "volcanic",
        "local_soil": "hard_rock",  # Dense basaltic lava bedrock
        "observed_felt_radius_km": "~100 - 150"
    },
    {
        "name": "2010 Christchurch (New Zealand)",
        "magnitude": 6.2,
        "depth_km": 5.0,
        "region": "active",
        "local_soil": "soft_basin",  # Canterbury alluvial gravels/silt
        "observed_felt_radius_km": "~250 - 350"
    }
]

TARGET_MMI = 2.5
results = []

for eq in earthquake_catalog:
    # 1. Baseline Engineering Rock (Vs30 = 760 m/s)
    r_rock = compute_felt_radius_v3(
        eq["magnitude"], eq["depth_km"], TARGET_MMI, eq["region"], soil_type="solid_rock",
        site_scaling_method="tanh"
    )
    
    # 2. Local Soil Profile Amplified Radius
    r_soil = compute_felt_radius_v3(
        eq["magnitude"], eq["depth_km"], TARGET_MMI, eq["region"], soil_type=eq["local_soil"],
        site_scaling_method="tanh"
    )
    
    results.append({
        "Earthquake Event": eq["name"],
        "Mag": eq["magnitude"],
        "Region": eq["region"],
        "Site Profile": eq["local_soil"],
        "Rock Radius (km)": round(r_rock, 1),
        "Soil Radius (km)": round(r_soil, 1),
        "Observed USGS Radius (km)": eq["observed_felt_radius_km"]
    })

df_results = pd.DataFrame(results)

print(f"=== EARTHQUAKE FELT RADIUS VALIDATION WITH SOIL AMPLIFICATION (Target MMI = {TARGET_MMI}) ===")
print(df_results.to_string(index=False))

=== EARTHQUAKE FELT RADIUS VALIDATION WITH SOIL AMPLIFICATION (Target MMI = 2.5) ===
               Earthquake Event  Mag      Region Site Profile  Rock Radius (km)  Soil Radius (km) Observed USGS Radius (km)
            2015 Gorkha (Nepal)  7.8      active   soft_basin             833.2             929.9               ~800 - 1000
   2011 Mineral (Virginia, USA)  5.8 stable_cena   stiff_soil            1433.9            1680.8               ~800 - 1000
            2011 Tohoku (Japan)  9.1  subduction   soft_basin            1629.9            1792.2                    > 1200
    2023 Kahramanmaraş (Turkey)  7.8 strike_slip   stiff_soil             604.9             655.9                ~700 - 900
  1989 Loma Prieta (California)  6.9 strike_slip   soft_basin             392.5             452.1                ~350 - 450
  2021 Fagradalsfjall (Iceland)  5.7    volcanic    hard_rock             122.0             110.5                ~100 - 150
2010 Christchurch (New Zealand)  6.2      activ